# 

In [2]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list, system_prompt
import selfies as sf
import re

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
smol_dataset = load_dataset(
    "osunlp/SMolInstruct",
    use_selfies=True,
    insert_core_tags=False,  # loada data w/o core tags such as <SELFIES>, </SELFIES>
    trust_remote_code=True,
)

In [3]:
def get_smol_data_list(
        data,
        task,
        instruction_templates,
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(data)))
    for i in iter_bar:
        data_instance = data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    list_tr_data = get_data_list(
        list_mol=list_tr_mol,
        list_label=list_tr_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    list_te_data = get_data_list(
        list_mol=list_te_mol,
        list_label=list_te_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    print(len(list_tr_data), len(list_te_data))
    return list_tr_data, list_te_data

In [4]:
task = "smol-property_prediction-esol"
instruction_templates = instructions_smol.property_prediction_esol_used_later

_task = re.sub("smol-", "", task)  # remove smol- from smol-<task_name>

# DEBUG: to avoid lengthy processing time
train_dataset = smol_dataset["train"].filter(lambda x: x["task"] == _task)
test_dataset = smol_dataset["test"].filter(lambda x: x["task"] == _task)

list_train_selfies = train_dataset['raw_input']
list_test_selfies = test_dataset['raw_input']

list_train_smiles = [sf.decoder(i) for i in list_train_selfies]
list_test_smiles = [sf.decoder(i) for i in list_test_selfies]

list_train_mol = [Chem.MolFromSmiles(i) for i in list_train_smiles]
list_test_mol = [Chem.MolFromSmiles(i) for i in list_test_smiles]

list_train_label = train_dataset['raw_output']
list_test_label = test_dataset['raw_output']


list_train_esol_data = get_data_list(
    list_mol=list_train_mol,
    list_label=list_train_label,
    task=task,
    instruction_templates=instruction_templates,
)
list_test_esol_data = get_data_list(
    list_mol=list_test_mol,
    list_label=list_test_label,
    task=task,
    instruction_templates=instruction_templates,
)

data_dict = {
    "train": list_train_esol_data,
    "test": list_test_esol_data
}

for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 112/112 [00:00<00:00, 4712.80 examples/s]


In [5]:
task = "smol-property_prediction-lipo"
instruction_templates = instructions_smol.property_prediction_lipo

_task = re.sub("smol-", "", task)  # remove smol- from smol-<task_name>

train_dataset = smol_dataset["train"].filter(lambda x: x["task"] == _task)
test_dataset = smol_dataset["test"].filter(lambda x: x["task"] == _task)

list_train_selfies = train_dataset['raw_input']
list_train_selfies = [i.replace(';', '.') for i in list_train_selfies]
list_test_selfies = test_dataset['raw_input']
list_test_selfies = [i.replace(';', '.') for i in list_test_selfies]

list_train_smiles = [sf.decoder(i) for i in list_train_selfies]
list_test_smiles = [sf.decoder(i) for i in list_test_selfies]

list_train_mol = [Chem.MolFromSmiles(i) for i in list_train_smiles]
list_test_mol = [Chem.MolFromSmiles(i) for i in list_test_smiles]

list_train_label = train_dataset['raw_output']
list_test_label = test_dataset['raw_output']


list_train_esol_data = get_data_list(
    list_mol=list_train_mol,
    list_label=list_train_label,
    task=task,
    instruction_templates=instruction_templates,
)
list_test_esol_data = get_data_list(
    list_mol=list_test_mol,
    list_label=list_test_label,
    task=task,
    instruction_templates=instruction_templates,
)

data_dict = {
    "train": list_train_esol_data,
    "test": list_test_esol_data
}

for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 420/420 [00:00<00:00, 6593.46 examples/s]


In [6]:
task = "smol-property_prediction-bbbp"
instruction_templates = instructions_smol.property_prediction_bbbp

_task = re.sub("smol-", "", task)  # remove smol- from smol-<task_name>

# DEBUG: to avoid lengthy processing time
train_dataset = smol_dataset["train"].filter(lambda x: x["task"] == _task)
test_dataset = smol_dataset["test"].filter(lambda x: x["task"] == _task)

list_train_selfies = train_dataset['raw_input']
list_train_selfies = [i.replace(';', '.') for i in list_train_selfies]
list_test_selfies = test_dataset['raw_input']
list_test_selfies = [i.replace(';', '.') for i in list_test_selfies]

list_train_smiles = [sf.decoder(i) for i in list_train_selfies]
list_test_smiles = [sf.decoder(i) for i in list_test_selfies]

list_train_mol = [Chem.MolFromSmiles(i) for i in list_train_smiles]
list_test_mol = [Chem.MolFromSmiles(i) for i in list_test_smiles]

list_train_label = train_dataset['raw_output']
list_test_label = test_dataset['raw_output']


list_train_esol_data = get_data_list(
    list_mol=list_train_mol,
    list_label=list_train_label,
    task=task,
    instruction_templates=instruction_templates,
)
list_test_esol_data = get_data_list(
    list_mol=list_test_mol,
    list_label=list_test_label,
    task=task,
    instruction_templates=instruction_templates,
)

data_dict = {
    "train": list_train_esol_data,
    "test": list_test_esol_data
}

for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_0219"
    )

[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not removing hydrogen atom without neighbors
[07:00:22] WARNING: not r

In [7]:
task = "smol-property_prediction-clintox"
instruction_templates = instructions_smol.property_prediction_clintox

_task = re.sub("smol-", "", task)  # remove smol- from smol-<task_name>

# DEBUG: to avoid lengthy processing time
train_dataset = smol_dataset["train"].filter(lambda x: x["task"] == _task)
test_dataset = smol_dataset["test"].filter(lambda x: x["task"] == _task)

list_train_selfies = train_dataset['raw_input']
list_train_selfies = [i.replace(';', '.') for i in list_train_selfies]
list_test_selfies = test_dataset['raw_input']
list_test_selfies = [i.replace(';', '.') for i in list_test_selfies]

list_train_smiles = [sf.decoder(i) for i in list_train_selfies]
list_test_smiles = [sf.decoder(i) for i in list_test_selfies]

list_train_mol = [Chem.MolFromSmiles(i) for i in list_train_smiles]
list_test_mol = [Chem.MolFromSmiles(i) for i in list_test_smiles]

list_train_label = train_dataset['raw_output']
list_test_label = test_dataset['raw_output']


list_train_esol_data = get_data_list(
    list_mol=list_train_mol,
    list_label=list_train_label,
    task=task,
    instruction_templates=instruction_templates,
)
list_test_esol_data = get_data_list(
    list_mol=list_test_mol,
    list_label=list_test_label,
    task=task,
    instruction_templates=instruction_templates,
)

data_dict = {
    "train": list_train_esol_data,
    "test": list_test_esol_data
}

for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 144/144 [00:00<00:00, 4215.20 examples/s]


In [8]:
task = "smol-property_prediction-hiv"
instruction_templates = instructions_smol.property_prediction_hiv

_task = re.sub("smol-", "", task)  # remove smol- from smol-<task_name>

# DEBUG: to avoid lengthy processing time
train_dataset = smol_dataset["train"].filter(lambda x: x["task"] == _task)
test_dataset = smol_dataset["test"].filter(lambda x: x["task"] == _task)

list_train_selfies = train_dataset['raw_input']
list_train_selfies = [i.replace(';', '.') for i in list_train_selfies]
list_test_selfies = test_dataset['raw_input']
list_test_selfies = [i.replace(';', '.') for i in list_test_selfies]

list_train_smiles = [sf.decoder(i) for i in list_train_selfies]
list_test_smiles = [sf.decoder(i) for i in list_test_selfies]

list_train_mol = [Chem.MolFromSmiles(i) for i in list_train_smiles]
list_test_mol = [Chem.MolFromSmiles(i) for i in list_test_smiles]

list_train_label = train_dataset['raw_output']
list_test_label = test_dataset['raw_output']


list_train_esol_data = get_data_list(
    list_mol=list_train_mol,
    list_label=list_train_label,
    task=task,
    instruction_templates=instruction_templates,
)
list_test_esol_data = get_data_list(
    list_mol=list_test_mol,
    list_label=list_test_label,
    task=task,
    instruction_templates=instruction_templates,
)

data_dict = {
    "train": list_train_esol_data,
    "test": list_test_esol_data
}

for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_0219"
    )

[07:00:48] WARNING: not removing hydrogen atom without neighbors
[07:00:48] WARNING: not removing hydrogen atom without neighbors
Saving the dataset (1/1 shards): 100%|██████████| 4107/4107 [00:00<00:00, 9840.67 examples/s]


In [10]:
test_dataset['raw_output']

['Yes',
 'No',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'Yes

In [12]:
test_dataset[0]

{'sample_id': 'property_prediction-sider.test.0.Hepatobiliary disorders',
 'input': 'Can [N][C][=C][C][=C][Branch1][O][N][=N][C][=C][C][=C][C][=C][Ring1][=Branch1][C][Branch1][C][N][=N][Ring1][#C] cause hepatobiliary disorders?',
 'output': 'Yes',
 'raw_input': '[N][C][=C][C][=C][Branch1][O][N][=N][C][=C][C][=C][C][=C][Ring1][=Branch1][C][Branch1][C][N][=N][Ring1][#C]',
 'raw_output': 'Yes',
 'split': 'test',
 'task': 'property_prediction-sider',
 'input_core_tag_left': '<SELFIES>',
 'input_core_tag_right': '</SELFIES>',
 'output_core_tag_left': '<BOOLEAN>',
 'output_core_tag_right': '</BOOLEAN>',
 'target': 'Hepatobiliary disorders'}

In [4]:
task = "smol-property_prediction-sider"
# In case of sider, which is multi-class classification dataset, we need to use the property specific templates
# instead, just use 5M dataset and filter sider tasks
trainset = datasets.load_from_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train')
testset = datasets.load_from_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test')

sider_trainset = trainset.filter(
    lambda x:"sider" in x["task"],
    num_proc=200
)
sider_testset = testset.filter(
    lambda x:"sider" in x["task"],
    num_proc=200
)


Filter (num_proc=200): 100%|██████████| 32851/32851 [00:01<00:00, 21640.25 examples/s]


In [10]:
sider_trainset.save_to_disk(
    f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_{task}_0219"
)
sider_testset.save_to_disk(
    f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_{task}_0219"
)
sider_testset.save_to_disk(
    f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_validation_{task}_0219"
)

Saving the dataset (1/1 shards): 100%|██████████| 2860/2860 [00:00<00:00, 7824.25 examples/s] 


In [5]:
sider_trainset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 22820
})

In [7]:
sider_trainset[0]

{'task': 'smol-property_prediction-sider',
 'x': [[7, 0, 2, 5, 1, 0, 2, 0, 0],
  [5, 0, 4, 5, 1, 0, 2, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 4, 5, 0, 0, 2, 0, 0],
  [8, 0, 1, 5, 0, 0, 2, 0, 0],
  [8, 0, 1, 5, 0, 0, 2, 0, 0],
  [8, 0, 1, 5, 0, 0, 2, 0, 0],
  [6, 0, 2, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 4, 5, 0, 0, 2, 0, 0],
  [8, 0, 1, 5, 0, 0, 2, 0, 0],
  [8, 0, 1, 5, 0, 0, 2, 0, 0],
  [8, 0, 1, 5, 0, 0, 2, 0, 0],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [6, 0, 3, 5, 1, 0, 2, 0, 1]],
 'edge_index': [[0,
   1,
   1,
   2,
   2,
   3,
   3,
   4,
   4,
   5,
   5,
   6,
   5,
   7,
   5,
   8,
   4,
   9,
   9,
   10,
   10,
   

In [6]:
sider_testset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 2860
})